# Make figure 6 of the paper
References: 
- https://www.pygmt.org/v0.2.0/gallery/plot/meca.html

In [ ]:
import pandas as pd
import pygmt 
from pyproj import Transformer
import matplotlib.pyplot as plt
from obspy.core.event import read_events

import numpy as np
import matplotlib.colors as mcolors
from matplotlib import cm



In [ ]:
df = pd.read_csv('../data/datasets_all_regions/origin_2010_2015_reloc_cog_ver3_cc.csv',index_col=0)
df


In [ ]:
_df = df[(df.rms < 2.5) & (df.p_picks >= 4) & (df.s_picks >= 4)].sort_values('nass', ascending=True)
_df

## Entire Region

In [ ]:


# # Initialize the figure
# fig = pygmt.Figure()

# # Define target region
# min_lat = 39
# max_lat = 51
# min_lon = -130
# max_lon = -121

# # Define the region to plot
# region = [min_lon, max_lon, min_lat, max_lat]
# title = "(b)"

# # Load grid data
# grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
# shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# # Set up the base map
# fig.basemap(region=region, projection="M12c", frame="af")

# fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
# fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# # Apply logarithmic scale for color mapping
# # Apply log10 transformation to nass values and set color palette
# nass_log_min = _df['nass'].min()   # Add 1 to avoid log(0) error
# nass_log_max = _df['nass'].max() 
# pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])

# # Plot a dummy point to add the station legend
# fig.plot(x=[-110], y=[48], style="c0.07c", fill="white", label="Catalog Event", pen="0.2p,black")
# # Plot the stations with the color map applied to the logarithm of "nass"
# fig.plot(
#     x=_df["lon"],
#     y=_df["lat"],
#     style="c0.07c",
#     fill=_df["nass"] ,  # Apply log transformation for color mapping
#     cmap=True,
#     pen="0.1p,black",
#     transparency=30
# )

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lLog(Number of P and S Picks)"],
# )
# # Save the figure
# fig.savefig("../data/datasets_all_regions/all_catalog_events_2010_2015_ver3_nass_log.pdf")

# # Display the figure
# fig.show()


## Plot with the focal mechanisms

In [ ]:
usgs_cat = read_events('../data/datasets_all_regions/Cascadia_momenttensors_M5_9.xml')

In [ ]:
longitude = []
latitude = []
depth = []
strike = []
dip = []
rake = []
magnitudes = []

for event in usgs_cat:
    try:
        lon = event.origins[1].longitude
        lat = event.origins[1].latitude
        z = event.origins[1].depth / 1000  # Convert depth to km
        for i in range(len(event.focal_mechanisms)):
            if event.focal_mechanisms[i].nodal_planes:
                nodal_plane = event.focal_mechanisms[i].nodal_planes.nodal_plane_1
                continue
            if not event.focal_mechanisms[i].nodal_planes:
                continue
        mag = event.preferred_magnitude()['mag']
        sdr = [nodal_plane.strike, nodal_plane.dip, nodal_plane.rake]
        longitude.append(lon)
        latitude.append(lat)
        depth.append(z)
        strike.append(sdr[0])
        dip.append(sdr[1])
        rake.append(sdr[2])
        magnitudes.append(mag)

    except (IndexError, AttributeError) as e:
        print(f"Skipping event due to missing data: {e}")

df = pd.DataFrame({'lon': longitude, 'lat': latitude, 'strike': strike, 'dip': dip, 'rake': rake,'depth': depth,'mag':magnitudes})


In [ ]:
len(usgs_cat)

In [ ]:
df

In [ ]:
df['mag'] = 5.5

In [ ]:
# # %matplotlib widget
# lon = []
# lat = []
# strike = []
# dip = []
# rake = []
# magnitude = []
# depth = []
# for c in usgs_cat:
#     lon.append(c.origins[0].longitude)
#     lat.append(c.origins[0].latitude)
    
#     nodalplane = c.focal_mechanisms[1].nodal_planes.nodal_plane_1
#     sdr = [nodalplane.strike,nodalplane.dip,nodalplane.rake]
#     mag = c.preferred_magnitude().mag
#     depth = c.origins[0].depth / 1000.0  # Convert depth from meters to kilometers
#     magnitude.append(mag)
#     strike.append(sdr[0])
#     dip.append(sdr[1])
#     rake.append(sdr[2])

# df = pd.DataFrame({'lon': lon, 'lat': lat, 'strike': strike, 'dip': dip, 'rake': rake, 'mag': mag, 'depth': depth})
# # Plot the data

In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 39
max_lat = 51
min_lon = -130
max_lon = -121

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()   # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max() 
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])

# Plot a dummy point to add the station legend
fig.plot(x=[-110], y=[48], style="c0.07c", fill="white", label="Catalog Event", pen="0.2p,black")
# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.07c",
    fill=_df["nass"] ,  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=30
)

focal_mechanism = dict(
    strike=df.strike,
    dip=df.dip,
    rake=df.rake,
    magnitude=df.mag  # Depth in km
)

offset_beachball = 0.3
# pass the focal mechanism data to meca in addition to the scale and event location
fig.meca(focal_mechanism, scale="0.3c", longitude=df.lon, latitude=df.lat, depth=df.depth,plot_latitude=df.lat+offset_beachball,plot_longitude=df.lon+offset_beachball,offset="0.2p,orange+s0.07c")

# Add title text
fig.colorbar(
    position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
    box="+gwhite@30+p0.8p,black",
    frame=["x+lLog(Number of P and S Picks)"],
)
# Save the figure
fig.savefig("../data/datasets_all_regions/all_catalog_events_2010_2015_ver3_cc_nass_fo_m4.png")

# Display the figure
fig.show()


## Axial and Endeavor


In [ ]:
# Drawn at its printed size: the manuscript places this map at 0.8 x 8.6 cm = 6.9 cm wide,
# so fonts, markers and pens below are the sizes on the page.
MAP_W = "6.9c"
proj = f"M{MAP_W}"

# Initialize the figure
fig = pygmt.Figure()
pygmt.config(FONT_ANNOT_PRIMARY="7p,Helvetica,black", FONT_LABEL="7p,Helvetica,black",
             MAP_FRAME_PEN="0.5p,black", MAP_TICK_LENGTH_PRIMARY="0.08c",
             MAP_ANNOT_OFFSET_PRIMARY="0.06c", MAP_LABEL_OFFSET="0.08c",
             MAP_FRAME_TYPE="plain")

# Define target region
min_lat = 46
max_lat = 50.5
min_lon = -130
max_lon = -126

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection=proj, frame=["WSne", "xa1f0.5", "ya1f0.5"])

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection=proj, shorelines="0.25p,black", borders=["1/0.3p", "2/0.2p"], resolution="f")

# Add plate boundaries
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="0.8p,dodgerblue1", transparency=50)

# Marker color = nass, the number of associated P and S picks (nass == p_picks + s_picks),
# on a linear scale between the catalog min and max; viridis (perceptually uniform)
nass_min = _df['nass'].min()
nass_max = _df['nass'].max()
pygmt.makecpt(cmap="viridis", series=[nass_min, nass_max])

fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.07c",
    fill=_df["nass"],
    cmap=True,
    pen="0.05p,black",
    transparency=35
)
# Dummy objects (off-map) for the legend
fig.plot(x=[-110], y=[48], style="c0.07c", fill="white", label="Catalog event", pen="0.2p,black")
fig.plot(x=[120, +120], y=[-30, -30], pen="0.8p,dodgerblue1", label="Plate boundary")

fig.legend(position="jTR+o0.1c", box="+gwhite+p0.3p,black")

# Horizontal colorbar below the map. GMT's modern theme shrinks colorbar fonts by
# sqrt(bar length / 15 cm) (0.58 for a 5 cm bar), so request 7p / 0.577 = 12.1p to print at 7p.
CB_LEN = 5.0
cb_font = f"{7 / (CB_LEN / 15) ** 0.5:.1f}p,Helvetica,black"
with pygmt.config(FONT_ANNOT_PRIMARY=cb_font, FONT_LABEL=cb_font):
    fig.colorbar(
        position=f"JBC+o0c/0.7c+h+w{CB_LEN}c/0.18c",
        frame=["xa50f25+lNumber of P and S picks per event"],
    )

fig.savefig("../data/datasets_all_regions/axial_endeavor_events_2010_2015_ver3_cc_nass.png", dpi=600)

# Display the figure
fig.show()

## With focal mechanism

In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 46
max_lat = 50.5
min_lon = -130
max_lon = -126

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]


# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()  # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])
title = "(a)"

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=35
)
# Plot a dummy objects for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
        x=[120, +120],
        y=[-30, -30],
        pen="1.5p,dodgerblue1",label='Plate Boundaries'
    )

# Position the text 1.5 cm from the top left corner
fig.text(
    x=min_lon + (max_lon - min_lon) * (1 / 15),  # Adjust x based on 1.5 cm from left
    y=max_lat - (max_lat - min_lat) * (1 / 28),  # Adjust y based on 1.5 cm from top
    text=title,
    font="25p,Helvetica-Bold,black",
    no_clip=True
)

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )
focal_mechanism = dict(
    strike=df.strike,
    dip=df.dip,
    rake=df.rake,
    magnitude=df.mag  # Depth in km
)

offset_beachball = 0.3
# pass the focal mechanism data to meca in addition to the scale and event location
fig.meca(focal_mechanism, scale="0.55c", longitude=df.lon, latitude=df.lat, depth=df.depth,plot_latitude=df.lat+offset_beachball,plot_longitude=df.lon+offset_beachball,offset="0.2p,black+s0.12c")
# Add the legend to the plot
# fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')
# Save the figure
fig.savefig("../data/datasets_all_regions/axial_endeavor_events_2010_2015_ver3_cc_nass_fo_4_no_legend.png")

# Display the figure
fig.show()


## Mendicino


In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 40
max_lat = 43
min_lon = -128
max_lon = -122

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"


# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()  # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])
title = "(b)"

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=35
)
# Plot a dummy objects for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
        x=[120, +120],
        y=[-30, -30],
        pen="1.5p,dodgerblue1",label='Plate Boundaries'
    )

# Position the text 1.5 cm from the top left corner
# fig.text(
#     x=min_lon + (max_lon - min_lon) * (1 / 15),  # Adjust x based on 1.5 cm from left
#     y=max_lat - (max_lat - min_lat) * (1 / 13),  # Adjust y based on 1.5 cm from top
#     text=title,
#     font="25p,Helvetica-Bold,black",
#     no_clip=True
# )

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )

# Add the legend to the plot
fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')
# Save the figure
fig.savefig("../data/datasets_all_regions/mendocino_events_2010_2015_ver3_cc_nass_no_relief_no_label.png")

# Display the figure
fig.show()


## With focal mechanism

In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 40
max_lat = 43
min_lon = -128
max_lon = -122

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"


# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()  # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])
title = "(b)"

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=35
)
# Plot a dummy objects for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
        x=[120, +120],
        y=[-30, -30],
        pen="1.5p,dodgerblue1",label='Plate Boundaries'
    )

# Position the text 1.5 cm from the top left corner
# Position the text 1.5 cm from the top left corner
fig.text(
    x=min_lon + (max_lon - min_lon) * (1 / 15),  # Adjust x based on 1.5 cm from left
    y=max_lat - (max_lat - min_lat) * (1 / 15),  # Adjust y based on 1.5 cm from top
    text=title,
    font="25p,Helvetica-Bold,black",
    no_clip=True
)

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )
focal_mechanism = dict(
    strike=df.strike,
    dip=df.dip,
    rake=df.rake,
    magnitude=df.mag  # Depth in km
)

offset_beachball = 0.3
# pass the focal mechanism data to meca in addition to the scale and event location
fig.meca(focal_mechanism, scale="0.55c", longitude=df.lon, latitude=df.lat, depth=df.depth,plot_latitude=df.lat+offset_beachball,plot_longitude=df.lon+offset_beachball,offset="0.2p,black+s0.12c")
# Add the legend to the plot
# fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')
# Save the figure
fig.savefig("../data/datasets_all_regions/mendocino_events_2010_2015_ver3_cc_nass_fo_4_no_legend.png")

# Display the figure
fig.show()


## A zoomed-in view of the junction

In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 40
max_lat = 41
min_lon = -125
max_lon = -124


# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"


# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()  # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])
title = ""

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=35
)
# Plot a dummy objects for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
        x=[120, +120],
        y=[-30, -30],
        pen="1.5p,dodgerblue1",label='Plate Boundaries'
    )

fig.text(
    x=-125.25, y=40.36, text=title,
    font="25p,Helvetica-Bold,black",
    no_clip=True    # This prevents the text from being clipped
)

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )

# Add the legend to the plot
fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')
# Save the figure
fig.savefig("../data/datasets_all_regions/mendocino_events_zoomed_in_2010_2015_ver3_cc_nass_no_relief.png")

# Display the figure
fig.show()


## Around 44.5N


In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 43
max_lat = 46
min_lon = -127
max_lon = -123

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"


# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()  # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])
title = "(c)"

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=35
)
# Plot a dummy objects for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
        x=[120, +120],
        y=[-30, -30],
        pen="1.5p,dodgerblue1",label='Plate Boundaries'
    )

# Position the text 1.5 cm from the top left corner
# fig.text(
#     x=min_lon + (max_lon - min_lon) * (1 / 15),  # Adjust x based on 1.5 cm from left
#     y=max_lat - (max_lat - min_lat) * (1 / 19),  # Adjust y based on 1.5 cm from top
#     text=title,
#     font="25p,Helvetica-Bold,black",
#     no_clip=True
# )

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )

# Add the legend to the plot
fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')
# Save the figure
fig.savefig("../data/datasets_all_regions/around_445_events_2010_2015_ver3_cc_nass_no_relief_no_label.png")

# Display the figure
fig.show()


## With focal mechanism

In [ ]:


# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 43
max_lat = 46
min_lon = -127
max_lon = -123

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"


# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
# Apply log10 transformation to nass values and set color palette
nass_log_min = _df['nass'].min()  # Add 1 to avoid log(0) error
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])
title = "(c)"

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],  # Apply log transformation for color mapping
    cmap=True,
    pen="0.1p,black",
    transparency=35
)
# Plot a dummy objects for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
        x=[120, +120],
        y=[-30, -30],
        pen="1.5p,dodgerblue1",label='Plate Boundaries'
    )

# Position the text 1.5 cm from the top left corner
## Position the text 1.5 cm from the top left corner
fig.text(
    x=min_lon + (max_lon - min_lon) * (1 / 15),  # Adjust x based on 1.5 cm from left
    y=max_lat - (max_lat - min_lat) * (1 / 20),  # Adjust y based on 1.5 cm from top
    text=title,
    font="28p,Helvetica-Bold,black",
    no_clip=True
)

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )
focal_mechanism = dict(
    strike=df.strike,
    dip=df.dip,
    rake=df.rake,
    magnitude=df.mag  # Depth in km
)

offset_beachball = 0.3
# pass the focal mechanism data to meca in addition to the scale and event location
fig.meca(focal_mechanism, scale="0.55c", longitude=df.lon, latitude=df.lat, depth=df.depth,plot_latitude=df.lat+offset_beachball,plot_longitude=df.lon+offset_beachball,offset="0.2p,black+s0.12c")
# Add the legend to the plot
# fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')
# Save the figure
fig.savefig("../data/datasets_all_regions/around_445_events_2010_2015_ver3_cc_nass_fo_4_no_legend.png")

# Display the figure
fig.show()


## Northern WA

In [ ]:
# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 46
max_lat = 50
min_lon = -128
max_lon = -123

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(d)"

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
nass_log_min = _df['nass'].min()
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],
    cmap=True,
    pen="0.1p,black",
    transparency=35
)

# Plot a dummy object for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
    x=[120, +120],
    y=[-30, -30],
    pen="1.5p,dodgerblue1", label='Plate Boundaries'
)

# Position the text 1.5 cm from the top left corner
# fig.text(
#     x=min_lon + (max_lon - min_lon) * (1 / 14),  # Adjust x based on 1.5 cm from left
#     y=max_lat - (max_lat - min_lat) * (1 / 21),  # Adjust y based on 1.5 cm from top
#     text=title,
#     font="25p,Helvetica-Bold,black",
#     no_clip=True
# )

# Add the legend to the plot
fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')

# Save the figure
fig.savefig("../data/datasets_all_regions/northern_wa_events_2010_2015_ver3_cc_nass_no_relief_no_label.png")

# Display the figure
fig.show()

## With focal mechanism

In [ ]:
# Initialize the figure
fig = pygmt.Figure()

# Define target region
min_lat = 46
max_lat = 50
min_lon = -128
max_lon = -123

# Define the region to plot
region = [min_lon, max_lon, min_lat, max_lat]
title = "(d)"

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Set up the base map
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Add plate boundaries 
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.5p,dodgerblue1", transparency=50)

# Apply logarithmic scale for color mapping
nass_log_min = _df['nass'].min()
nass_log_max = _df['nass'].max()
pygmt.makecpt(cmap="plasma", series=[nass_log_min, nass_log_max])

# Plot the stations with the color map applied to the logarithm of "nass"
fig.plot(
    x=_df["lon"],
    y=_df["lat"],
    style="c0.12c",
    fill=_df["nass"],
    cmap=True,
    pen="0.1p,black",
    transparency=35
)

# Plot a dummy object for the legend
fig.plot(x=[-110], y=[48], style="c0.12c", fill="white", label="Catalog Event", pen="0.2p,black")

fig.plot(
    x=[120, +120],
    y=[-30, -30],
    pen="1.5p,dodgerblue1", label='Plate Boundaries'
)

# Position the text 1.5 cm from the top left corner
# Position the text 1.5 cm from the top left corner
fig.text(
    x=min_lon + (max_lon - min_lon) * (1 / 15),  # Adjust x based on 1.5 cm from left
    y=max_lat - (max_lat - min_lat) * (1 / 28),  # Adjust y based on 1.5 cm from top
    text=title,
    font="25p,Helvetica-Bold,black",
    no_clip=True
)

# # Add title text
# fig.colorbar(
#     position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
#     box="+gwhite@30+p0.8p,black",
#     frame=["x+lNumber of P and S Picks"],
# )
focal_mechanism = dict(
    strike=df.strike,
    dip=df.dip,
    rake=df.rake,
    magnitude=df.mag  # Depth in km
)

offset_beachball = 0.3
# pass the focal mechanism data to meca in addition to the scale and event location
fig.meca(focal_mechanism, scale="0.55c", longitude=df.lon, latitude=df.lat, depth=df.depth,plot_latitude=df.lat+offset_beachball,plot_longitude=df.lon+offset_beachball,offset="0.2p,black+s0.12c")
# Add the legend to the plot
# fig.legend(position="JTR+w3.7+jTR", box='+gwhite+p1p')


# Save the figure
fig.savefig("../data/datasets_all_regions/northern_wa_events_2010_2015_ver3_cc_nass_fo_4_no_legend.png")

# Display the figure
fig.show()

## Plot the colorbar

In [ ]:
# Define the normalization range and colormap
min_nass = _df['nass'].min()  # Replace with the appropriate data source if needed
max_nass = _df['nass'].max()

norm = mcolors.Normalize(vmin=min_nass, vmax=max_nass)
cmap = cm.plasma  # Use the same colormap as in the previous plots

# Create a figure for the color bar
fig, ax = plt.subplots(figsize=(6, 1))  # Adjust the size as needed
fig.subplots_adjust(bottom=0.5)

# Create the color bar
cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), cax=ax, orientation='horizontal')

# Set the color bar label
cbar.set_label('Number of P and S Picks', fontsize=15)

# Move the ticks and labels above the color bar
cbar.ax.xaxis.set_ticks_position('top')  # Move ticks to the top
cbar.ax.xaxis.set_label_position('top')  # Move the label to the top

# Save the color bar as a PNG
fig.savefig("../data/datasets_all_regions/colorbar.png", dpi=300, bbox_inches='tight')

# Display the color bar
plt.show()